# HPC Cluster Assistant — Rule-Based Chatbot with NLTK

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Japh-ai/HPC-chatbot-nltk/blob/main/notebooks/hpc_chatbot.ipynb)

**This is an academic project.**

A rule-based chatbot that answers frequently asked questions from users of a university HPC (High Performance Computing) cluster: connecting via SSH, submitting and managing SLURM jobs, loading software modules, and getting support.

**Pipeline:** user message → NLTK preprocessing (tokenize, lowercase, remove stopwords, stem) → keyword matching against the knowledge base (`data/intents.json`) → response.

## 1. Setup

In Google Colab, the repository is cloned so the knowledge base (`data/intents.json`) is available. Locally, the file is read from the project folder.

In [1]:
import sys
from pathlib import Path

# In Colab, download the repo so data/intents.json is available
if "google.colab" in sys.modules:
    !git clone -q https://github.com/Japh-ai/HPC-chatbot-nltk.git /content/HPC-chatbot-nltk
    INTENTS_PATH = Path("/content/HPC-chatbot-nltk/data/intents.json")
else:
    INTENTS_PATH = Path("../data/intents.json")

print("Python version:", sys.version.split()[0])

Python version: 3.12.14


In [2]:
import json
import random
import string

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Download NLTK resources (once)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

FALLBACK = "Sorry, I didn't understand that. Can you please rephrase?"
THRESHOLD = 0.3  # minimum match score to accept an intent

random.seed(42)  # reproducible responses for the test evidence
print("NLTK version:", nltk.__version__)

NLTK version: 3.10.4


## 2. Knowledge base

Each intent has a **tag** (internal name), **patterns** (example user phrases) and **responses** (what the bot replies).

In [11]:
def load_intents(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)["intents"]


intents = load_intents(INTENTS_PATH)
responses = {intent["tag"]: intent["responses"] for intent in intents}

print(f"{len(intents)} intents loaded\n")
print(f"{'tag':^22} {'patterns':^8} {'responses':^9}")
print("-" * 40)
for intent in intents:
    print(f"{intent['tag']:^22} {len(intent['patterns']):^8} {len(intent['responses']):^9}")

11 intents loaded

         tag           patterns responses
----------------------------------------
       greeting           3         1    
      submit_job          3         1    
       get_help           3         2    
   check_job_status       3         1    
      cancel_job          3         1    
       goodbye            3         1    
        thanks            3         1    
 job_submission_error     3         1    
     module_load          3         1    
     module_list          3         1    
  connect_to_cluster      3         1    


## 3. Text preprocessing (NLTK)

1. **Lowercase** the text.
2. **Tokenize** it into words with `word_tokenize`.
3. **Remove stopwords** (common words like *how, do, I, my*) and punctuation.
4. **Stem** each word with `PorterStemmer`, so *jobs* and *job*, or *running* and *run*, match.

In [12]:
def preprocess(text):
    """Lowercase, tokenize, remove stopwords/punctuation and stem."""
    tokens = word_tokenize(text.lower())
    return {
        stemmer.stem(t)
        for t in tokens
        if t not in stop_words and t not in string.punctuation
    }


for example in ["How do I check my job status?", "Is my job running?", "I want to cancel my jobs"]:
    print(f"{example!r:<35} -> {preprocess(example)}")

'How do I check my job status?'     -> {'job', 'statu', 'check'}
'Is my job running?'                -> {'job', 'run'}
'I want to cancel my jobs'          -> {'cancel', 'want', 'job'}


## 4. Intent detection (rules)

Each intent's patterns are merged into one set of keywords. For a user message, the score of an intent is:

$$score = \frac{\text{matching words}}{\text{words in the message}}$$

The intent with the highest score wins. If no intent reaches the threshold (0.3), the bot uses the fallback response.

In [13]:
def build_keywords(intents):
    """Turn each intent's patterns into one set of keywords."""
    return {
        intent["tag"]: set().union(*(preprocess(p) for p in intent["patterns"]))
        for intent in intents
    }


def detect_intent(text, keywords):
    """Return the tag whose keywords best match the text, or None."""
    words = preprocess(text)
    if not words:
        return None
    best_tag, best_score = None, 0
    for tag, intent_words in keywords.items():
        score = len(words & intent_words) / len(words)
        if score > best_score:
            best_tag, best_score = tag, score
    return best_tag if best_score >= THRESHOLD else None


def get_response(tag):
    return random.choice(responses[tag]) if tag else FALLBACK


keywords = build_keywords(intents)
for tag, words in keywords.items():
    print(f"{tag:<22} {sorted(words)}")

greeting               ['good', 'hello', 'hi', 'morn']
submit_job             ['cluster', 'job', 'run', 'sbatch', 'script', 'submit']
get_help               ['assist', 'get', 'help', 'need', 'support']
check_job_status       ['check', 'job', 'run', 'squeue', 'statu']
cancel_job             ['cancel', 'job', 'scancel', 'stop']
goodbye                ['bye', 'goodby', 'later', 'see']
thanks                 ['appreci', 'thank']
job_submission_error   ['error', 'fail', 'job', 'sbatch', 'submiss', 'submit']
module_load            ['load', 'modul', 'softwar']
module_list            ['avail', 'list', 'modul', 'see', 'softwar']
connect_to_cluster     ['cluster', 'connect', 'hpc', 'login', 'ssh']


## 5. Interactive chat

Run the cell and type your questions. Type `quit` to exit.

In [14]:
def chat():
    print("HPC Assistant is ready! Type 'quit' to exit.")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ("quit", "exit"):
            print("Bot: Goodbye!")
            break
        print("Bot:", get_response(detect_intent(user_input, keywords)))


chat()

HPC Assistant is ready! Type 'quit' to exit.
Bot: Goodbye!


In [15]:
keywords

{'greeting': {'good', 'hello', 'hi', 'morn'},
 'submit_job': {'cluster', 'job', 'run', 'sbatch', 'script', 'submit'},
 'get_help': {'assist', 'get', 'help', 'need', 'support'},
 'check_job_status': {'check', 'job', 'run', 'squeue', 'statu'},
 'cancel_job': {'cancel', 'job', 'scancel', 'stop'},
 'goodbye': {'bye', 'goodby', 'later', 'see'},
 'thanks': {'appreci', 'thank'},
 'job_submission_error': {'error',
  'fail',
  'job',
  'sbatch',
  'submiss',
  'submit'},
 'module_load': {'load', 'modul', 'softwar'},
 'module_list': {'avail', 'list', 'modul', 'see', 'softwar'},
 'connect_to_cluster': {'cluster', 'connect', 'hpc', 'login', 'ssh'}}

## 6. Functional tests

Ten conversations covering every intent, plus edge cases: synonyms, typos, out-of-scope questions and ambiguous messages. Each turn lists the message and the **expected** intent (`None` = the fallback should answer). The bot's detected intent is compared against it.

In [16]:
TEST_CONVERSATIONS = [
    ("1. Greeting and job submission", [
        ("hello", "greeting"),
        ("how do I submit a job?", "submit_job"),
        ("thanks", "thanks"),
        ("bye", "goodbye"),
    ]),
    ("2. Connecting to the cluster", [
        ("good morning", "greeting"),
        ("how can I login to the cluster?", "connect_to_cluster"),
        ("thank you", "thanks"),
    ]),
    ("3. Checking job status", [
        ("how do I check my job status?", "check_job_status"),
        ("squeue", "check_job_status"),
        ("is my job still running?", "check_job_status"),
    ]),
    ("4. Cancelling a job", [
        ("I want to stop my job", "cancel_job"),
        ("scancel", "cancel_job"),
        ("see you later", "goodbye"),
    ]),
    ("5. Software modules", [
        ("what software is available?", "module_list"),
        ("how do I load a module?", "module_load"),
        ("appreciate it", "thanks"),
    ]),
    ("6. Submission errors", [
        ("my sbatch gives an error", "job_submission_error"),
        ("my job failed to submit", "job_submission_error"),
        ("I need help", "get_help"),
    ]),
    ("7. Synonyms", [
        ("how do I kill a job?", "cancel_job"),
        ("can I view the queue?", "check_job_status"),
        ("hey there", "greeting"),
    ]),
    ("8. Typos", [
        ("helo", "greeting"),
        ("how do I sumbit a job", "submit_job"),
        ("scancell", "cancel_job"),
    ]),
    ("9. Out-of-scope questions", [
        ("what's the weather today?", None),
        ("tell me a joke", None),
        ("", None),
    ]),
    ("10. Full support session", [
        ("hi", "greeting"),
        ("how do I connect to the cluster?", "connect_to_cluster"),
        ("list modules", "module_list"),
        ("how do I run a script on the cluster?", "submit_job"),
        ("is my job running?", "check_job_status"),
        ("goodbye", "goodbye"),
    ]),
]


def run_conversation(title, turns):
    print(f"=== {title} ===")
    passed = 0
    for message, expected in turns:
        detected = detect_intent(message, keywords)
        ok = detected == expected
        passed += ok
        print(f"You: {message}")
        print(f"Bot: {get_response(detected)}")
        print(f"     [expected: {expected} | detected: {detected} | {'PASS' if ok else 'FAIL'}]")
    print()
    return passed, len(turns)


results = [(title, *run_conversation(title, turns)) for title, turns in TEST_CONVERSATIONS]

=== 1. Greeting and job submission ===
You: hello
Bot: Hi! I'm the HPC cluster assistant. How can I help you?
     [expected: greeting | detected: greeting | PASS]
You: how do I submit a job?
Bot: Create a SLURM script and submit it with: sbatch my_script.sh
     [expected: submit_job | detected: submit_job | PASS]
You: thanks
Bot: You're welcome! Happy to help.
     [expected: thanks | detected: thanks | PASS]
You: bye
Bot: Goodbye! If you have more questions, feel free to ask.
     [expected: goodbye | detected: goodbye | PASS]

=== 2. Connecting to the cluster ===
You: good morning
Bot: Hi! I'm the HPC cluster assistant. How can I help you?
     [expected: greeting | detected: greeting | PASS]
You: how can I login to the cluster?
Bot: You can connect to the HPC cluster using SSH: ssh username@cluster_address
     [expected: connect_to_cluster | detected: connect_to_cluster | PASS]
You: thank you
Bot: You're welcome! Happy to help.
     [expected: thanks | detected: thanks | PASS]

=

### Test summary

In [17]:
print(f"{'Conversation':<34} {'Passed':>6} {'Total':>5}")
for title, passed, total in results:
    print(f"{title:<34} {passed:>6} {total:>5}")

total_passed = sum(p for _, p, _ in results)
total_turns = sum(t for _, _, t in results)
print(f"\nAccuracy: {total_passed}/{total_turns} = {total_passed / total_turns:.0%}")

Conversation                       Passed Total
1. Greeting and job submission          4     4
2. Connecting to the cluster            3     3
3. Checking job status                  2     3
4. Cancelling a job                     3     3
5. Software modules                     3     3
6. Submission errors                    3     3
7. Synonyms                             0     3
8. Typos                                2     3
9. Out-of-scope questions               3     3
10. Full support session                5     6

Accuracy: 28/34 = 82%


## 7. Results analysis

### Results

The chatbot answered **28 of 34 messages correctly (82%)** across the 10 test conversations. It handled every in-scope question phrased similarly to its patterns, and it correctly rejected every out-of-scope message (weather, jokes, empty input) with the fallback response.

| Failed message | Expected | Detected | Cause |
|---|---|---|---|
| is my job running? | check_job_status | submit_job | Tie: *job* and *run* belong to both intents, and the first intent in the file wins. This message is an exact pattern from the knowledge base, and it still fails. |
| is my job still running? | check_job_status | submit_job | Same tie |
| how do I kill a job? | cancel_job | submit_job | *kill* is not in any pattern, so only *job* matches |
| can I view the queue? | check_job_status | fallback | *view* and *queue* are not in any pattern |
| hey there | greeting | fallback | *hey* is not in any pattern |
| helo | greeting | fallback | Typo, with no fuzzy matching |

Two passes were partly luck: *"how do I sumbit a job"* matched only through the same tie, and *"scancell"* matched because the stemmer happened to cut it down to *scancel*.

### Limitations of this first version

1. **Limited vocabulary.** The bot only recognizes words that appear in its patterns, so synonyms (*kill*, *view*, *hey*) fail. The synonym conversation scored 0/3.
2. **Shared keywords and ties.** Generic words like *job*, *run* and *cluster* appear in several intents. Every word counts the same, so a common word like *job* weighs as much as a specific one like *squeue*, and ties are broken by the order of the intents in the file.
3. **No typo tolerance.** A single misspelled letter (*helo*) is enough for a word to be missed.
4. **Stopword removal loses meaning.** NLTK's list includes negations (*not*, *don't*), so *"don't cancel my job"* is read the same as *"cancel my job"*.
5. **Word order is ignored.** Each message is treated as a set of words (bag of words).
6. **No conversational context.** Each message is handled on its own, so follow-ups like *"and how do I cancel it?"* cannot use the previous message.
7. **Static, generic responses.** The bot cannot access the real cluster (actual job status, quotas), and each intent has only 1–2 responses.
8. **Small, manually maintained knowledge base.** 11 intents with 3 patterns each, only in English.

### Improvements for the next stages

**Short term (still rule-based)**
- Add 5–10 patterns per intent with synonyms (*stop/kill/cancel*, *check/see/view*), and expand keywords automatically with WordNet (`nltk.corpus.wordnet`).
- Tolerate typos with fuzzy matching (`nltk.edit_distance`).
- Weight rare words higher than common ones (TF-IDF), and ask a clarifying question when two intents tie instead of picking the first one.
- Log unrecognized questions to find missing intents.

**Medium term**
- Replace the keyword rules with a machine learning intent classifier (e.g. TF-IDF + Logistic Regression or Naive Bayes with scikit-learn), trained on the patterns.
- Add conversational context (remember the last intent and entities such as a job ID).

**Long term**
- Use semantic embeddings or an LLM with retrieval (RAG) over the cluster's documentation, so the bot understands paraphrases.
- Offer a web or chat-platform interface (e.g. Streamlit, Slack) and Spanish support.